In [ ]:
from SPARQLWrapper import SPARQLWrapper, TURTLE, JSON, CSV
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob
import subprocess
import time
import numpy as np
import plotly.express as px

In [ ]:
dico_top_pathway_ids = {
    "01": "Autophagy",
    "02": "CellCycle",
    "03": "CellCellCommunication",
    "04": "CellularResponseToStimuli",
    "05": "ChromatinOrganization",
    "06": "CircadianClock",
    "07": "DevelopmentalBiology",
    "08": "DigestionAndAbsorption",
    "09": "Disease",
    "10": "DNARepair",
    "11": "DNAReplication",
    "12": "DrugADME",
    "13": "ExtracellularMatrixOrganization",
    "14": "GeneExpression(Transcription)",
    "15": "Hemostasis",
    "16": "ImmuneSystem",
    "17": "Metabolism",
    "18": "MetabolismOfProteins",
    "19": "MetabolismOfRNA",
    "20": "MuscleContraction",
    "21": "NeuronalSystem",
    "22": "OrganelleBiogenesisAndMaintenance",
    "23": "ProgrammedCellDeath",
    "24": "ProteinLocalization",
    "25": "Reproduction",
    "26": "SensoryPerception",
    "27": "SignalTransduction",
    "28": "TransportOfSmallMolecules",
    "29": "VesicleMediatedTransport"
}

In [ ]:
current_directory = os.getcwd()
BioPAX_Ontology_file_path = os.path.join(current_directory, "../../Data/BioPAX/BioPAXOntology/biopax-level3.owl")
BioPAX_files_path = os.path.join(current_directory, "../../Data/BioPAX/ReactomeTopPathways")
reactome_cliques_folder = os.path.join(current_directory, "../../Results/PathwayComembership/01_ComembershipCliques")
BioPAX_files = glob.glob(os.path.join(BioPAX_files_path, "*.xml"))
comembership_cliques = glob.glob(os.path.join(reactome_cliques_folder, "*.csv"))
weighted_comembership_cliques = glob.glob(os.path.join(current_directory, "../../Results/PathwayComembership/02_WeightedComembershipCliques/*.csv"))
print(weighted_comembership_cliques)

prefixes = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX bp3: <http://www.biopax.org/release/biopax-level3.owl#>
"""
endpoint = "http://localhost:3030/top_pathway"
print(BioPAX_files)

### Extract number of UniProt IDs per pathway

In [ ]:
# query = query = """ 
# SELECT (COUNT(DISTINCT ?id) AS ?nbID)
# WHERE {
#     VALUES ?db { "UniProt" "UniProt Isoform" } 
#     ?entityRef rdf:type/(rdfs:subClassOf*) bp3:ProteinReference .
#     ?entityRef bp3:xref ?entityRefXref .
#     ?entityRefXref bp3:db ?db .
#     ?entityRefXref bp3:id ?id .
# }
# """

# dico_pathway_nbUniprot = dict()

# counter = 1
# for owl_file in sorted(BioPAX_files):
#     print(owl_file)
#     command = [
#         '/home/cbeust/Softwares/JenaFuseki/apache-jena-fuseki-4.9.0/fuseki-server',
#         '--file', owl_file,
#         '--file', BioPAX_Ontology_file_path,
#         '/top_pathway'
#     ]
#     print("Fuseki command:", command)
#     process = subprocess.Popen(command)
#     time.sleep(30)
#     sparql = SPARQLWrapper(endpoint)
#     sparql.setQuery(prefixes + query)
#     sparql.setReturnFormat(JSON)
#     results = sparql.query().convert()
#     nbEr_total = int(results["results"]["bindings"][0]["nbID"]["value"])
#     dico_pathway_nbUniprot[f"{counter:02d}_toppathway"] = nbEr_total 
#     process.kill()
#     time.sleep(30)
#     counter += 1

# pathway_uniprot_df = pd.DataFrame(columns=["top_pathway", "nb_uniprot_ids"])
# counter = 0
# for key, value in dico_pathway_nbUniprot.items():
#     pathway_uniprot_df.at[counter, "top_pathway"] = dico_top_pathway_ids[key[0:2]]
#     pathway_uniprot_df.at[counter, "nb_uniprot_ids"] = value
#     counter += 1
# print(pathway_uniprot_df)
# pathway_uniprot_df.to_csv("top_pathways_uniprot_ids.csv", sep=",", header=1, index=False)

In [ ]:
pathway_uniprot_df = pd.read_csv("../../Results/PathwayComembership/OtherFiles/top_pathways_uniprot_ids.csv", sep=",", header=0)
print(pathway_uniprot_df.head())

dico_pathway_nbUniprot = dict()
for index, row in pathway_uniprot_df.iterrows():
    dico_pathway_nbUniprot[row[0]] = row[1]

print(dico_pathway_nbUniprot)

In [ ]:
bar_width = 0.4
x = np.arange(len(pathway_uniprot_df))
plt.figure(figsize=(10, 6))
plt.bar(x - bar_width/2, pathway_uniprot_df['nb_uniprot_ids'], width=bar_width, label='nb_uniprot_ids', color='blue')
plt.xticks(x, pathway_uniprot_df['top_pathway'], rotation=90)
plt.xlabel('Top Pathway')
plt.ylabel('Value')
plt.title('Number of uniprot IDS per top pathway')
plt.legend()

plt.tight_layout()
plt.show()

### Extract comembership cliques

In [ ]:
pathway_uniprot_df = pd.read_csv("../../Results/PathwayComembership/OtherFiles/top_pathways_uniprot_ids.csv", sep=",", header=0)
print(pathway_uniprot_df.head())

dico_pathway_nbUniprot = dict()
for index, row in pathway_uniprot_df.iterrows():
    dico_pathway_nbUniprot[row[0]] = row[1]

print(dico_pathway_nbUniprot)

In [ ]:
cliques_df = pd.DataFrame(columns=['top_pathway', 'nb_uniprot_ids', 'clique_size'])

i = 1
for file in sorted(comembership_cliques):
    clique = pd.read_csv(file, sep=",", header=0)
    clique_size = len(clique.index)
    toppathway_name = dico_top_pathway_ids[f"{i:02d}"]
    cliques_df.at[i, "top_pathway"] = toppathway_name
    cliques_df.at[i, "nb_uniprot_ids"] = dico_pathway_nbUniprot[dico_top_pathway_ids[f"{i:02d}"]]
    cliques_df.at[i, "clique_size"] = clique_size
    i += 1

print(cliques_df)
cliques_df.to_csv("../../Results/PathwayComembership/OtherFiles/reactome_cliques_size.csv", sep=",", header=1, index=False)

In [ ]:
bar_width = 0.4
x = np.arange(len(cliques_df))
plt.figure(figsize=(10, 6))
plt.bar(x + bar_width/2, cliques_df['clique_size'], width=bar_width, label='clique_size', color='red')
plt.xticks(x, cliques_df['top_pathway'], rotation=90)
plt.xlabel('Top Pathway')
plt.ylabel('Value')
plt.title('Number of uniprot IDS per top pathway and size of the comembership clique')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
weighted_clique_df = pd.DataFrame(columns=['top_pathway', 'nb_uniprot_ids', 'weighted_clique_size'])

i = 1
for file in sorted(weighted_comembership_cliques):
    weighted_clique = pd.read_csv(file, sep=",", header=0)
    weighted_clique_size = len(weighted_clique.index)
    toppathway_name = dico_top_pathway_ids[f"{i:02d}"]
    weighted_clique_df.at[i, "top_pathway"] = toppathway_name
    weighted_clique_df.at[i, "nb_uniprot_ids"] = dico_pathway_nbUniprot[dico_top_pathway_ids[f"{i:02d}"]]
    weighted_clique_df.at[i, "weighted_clique_size"] = weighted_clique_size
    i += 1

print(weighted_clique_df)
weighted_clique_df.to_csv("../../Results/PathwayComembership/OtherFiles/reactome_weighted_cliques_size.csv", sep=",", header=1, index=False)

In [ ]:
bar_width = 0.4
x = np.arange(len(weighted_clique_df))
plt.figure(figsize=(10, 6))
plt.bar(x + bar_width/2, weighted_clique_df['weighted_clique_size'], width=bar_width, label='clique_size', color='purple')
plt.xticks(x, weighted_clique_df['top_pathway'], rotation=90)
plt.xlabel('Top Pathway')
plt.ylabel('Value')
plt.title('Number of uniprot IDS per top pathway and size of the wieghted comembership clique')
plt.legend()

plt.tight_layout()
plt.show()

TODO: look at pathways that lost a lot of protein in the scoring phase (23_ProgrammedCellDeath, 18_MetabolismOfRNA)

### 3 - Analyze comembership cliques

In [ ]:
i = 1
for file in sorted(weighted_comembership_cliques):
    top_pathway_name = dico_top_pathway_ids[f"{i:02d}"]
    weighted_comembership_clique_df = pd.read_csv(file, sep=",", header=0)
    print(weighted_comembership_clique_df.head())
    fig = px.histogram(weighted_comembership_clique_df, x="scoreComembership", color="provenanceScore", title=f"Repartition and provenance of comembership scores for {top_pathway_name}")
    fig.show()
    i += 1